In [32]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms 
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [33]:
if torch.cuda.is_available():
  device = torch.device("cuda")
  print(f"Using GPU")
elif torch.backends.mps.is_available():
  device = torch.device("mps")
  print(f"Using Apple NPU")
else:
  device = torch.device("cpu")
  print(f"Using CPU")

Using GPU


In [34]:
train_transforms = transforms.Compose(
  [
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    transforms.RandomRotation(degrees=15),
  ]
)

test_transforms = transforms.Compose(
  [
    transforms.Resize((244, 244)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
  ]
)

In [35]:
full_dataset_for_train = torchvision.datasets.ImageFolder(
  root="labeled_data",
  transform=train_transforms
)

full_dataset_for_test = torchvision.datasets.ImageFolder(
  root="labeled_data",
  transform=test_transforms
)

idxs = np.arange(len(full_dataset_for_train))
labels = full_dataset_for_train.targets

train_idx, test_idx = train_test_split(
  idxs,
  test_size=0.2,
  stratify=labels
)

train_dataset = torch.utils.data.Subset(full_dataset_for_train, train_idx)
test_dataset = torch.utils.data.Subset(full_dataset_for_test, test_idx)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=2, shuffle=False)

In [36]:
np.sum(np.array(labels) == 0), np.sum(np.array(labels) == 1)

(np.int64(97), np.int64(31))

In [37]:
np.sum(np.array(labels) == 0) / len(labels), np.sum(np.array(labels) == 1) / len(labels)

(np.float64(0.7578125), np.float64(0.2421875))

In [38]:
model = nn.Sequential(
  nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7, padding="same"),
  nn.BatchNorm2d(num_features=64),
  nn.ReLU(),
  nn.MaxPool2d(kernel_size=2),
  nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding="same"),
  nn.BatchNorm2d(num_features=128),
  nn.ReLU(),
  nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding="same"),
  nn.BatchNorm2d(num_features=128),
  nn.ReLU(),
  nn.MaxPool2d(kernel_size=2),
  nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding="same"),
  nn.BatchNorm2d(num_features=256),
  nn.ReLU(),
  nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, padding="same"),
  nn.BatchNorm2d(num_features=256),
  nn.ReLU(),
  nn.MaxPool2d(kernel_size=2),
  nn.AdaptiveAvgPool2d(output_size=1),
  nn.Flatten(),
  nn.Linear(in_features=256, out_features=128), nn.ReLU(),
  nn.Linear(in_features=128, out_features=64), nn.ReLU(),
  nn.Linear(in_features=64, out_features=1)
).to(device)


In [39]:
m = nn.AdaptiveAvgPool2d(1)
input = torch.tensor([[[[1, 2], [3, 4]], [[5, 6], [7, 8]], [[9, 10], [11, 12]]], ], dtype=torch.float16)
m(input)

tensor([[[[ 2.5000]],

         [[ 6.5000]],

         [[10.5000]]]], dtype=torch.float16)

In [40]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

n_epochs = 100

In [41]:
model.train()

for epoch in range(n_epochs):
  running_loss = 0.0

  for i, data in enumerate(train_loader):
    inputs, labels = data[0].to(device), data[1].to(device).float().unsqueeze(1)

    optimizer.zero_grad()

    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()
  
  print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

Epoch 1, Loss: 0.6396368367999208
Epoch 2, Loss: 0.5866814501145307
Epoch 3, Loss: 0.5500202313357708
Epoch 4, Loss: 0.5323420576020783
Epoch 5, Loss: 0.5242864460337395
Epoch 6, Loss: 0.531931468669106
Epoch 7, Loss: 0.5506362903351877
Epoch 8, Loss: 0.5206288175255644
Epoch 9, Loss: 0.5067072130885779
Epoch 10, Loss: 0.5142061745419222
Epoch 11, Loss: 0.5062033890509138
Epoch 12, Loss: 0.4940970621856989
Epoch 13, Loss: 0.4816611455936058
Epoch 14, Loss: 0.5019336588242475
Epoch 15, Loss: 0.5107138262075537
Epoch 16, Loss: 0.48272940633343714
Epoch 17, Loss: 0.5057566615880704
Epoch 18, Loss: 0.47753405454112036
Epoch 19, Loss: 0.4379447973241993
Epoch 20, Loss: 0.4445534003715889
Epoch 21, Loss: 0.41536811929123074
Epoch 22, Loss: 0.45257215318726557
Epoch 23, Loss: 0.48278964267057534
Epoch 24, Loss: 0.4438396411783555
Epoch 25, Loss: 0.4343329434301339
Epoch 26, Loss: 0.44265609424488217
Epoch 27, Loss: 0.4482380952320847
Epoch 28, Loss: 0.48800798344845864
Epoch 29, Loss: 0.47534

In [49]:
all_predictions = []
all_labels = []

model.eval()

with torch.no_grad():
  for data in test_loader:
    inputs, labels = data[0].to(device), data[1].to(device).float().unsqueeze(1)
    outputs = model(inputs)
    predictions = torch.sigmoid(outputs) > 0.02
    all_predictions.extend(predictions.cpu().numpy())
    all_labels.extend(labels.cpu().numpy())

print(all_predictions)
print(all_labels)
print(np.sum(np.array(all_predictions) == np.array(all_labels).astype(bool)) / len(all_labels))

print(classification_report(all_labels, all_predictions))

[array([False]), array([ True]), array([False]), array([ True]), array([ True]), array([ True]), array([False]), array([ True]), array([ True]), array([False]), array([ True]), array([ True]), array([ True]), array([ True]), array([False]), array([ True]), array([False]), array([ True]), array([ True]), array([ True]), array([ True]), array([ True]), array([False]), array([ True]), array([False]), array([ True])]
[array([0.], dtype=float32), array([1.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([1.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([1.], dtype=float32), array([1.], dtype=float32), array([0.], dtype=floa